<a href="https://colab.research.google.com/github/Monthe5/Portfolio-Projects/blob/main/Catering_Decision_Theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem Defination**
The business under cosideration is a catering business which sources products from various suppliers and need to decide which suplier is best for every category and which mode of transportation to use. The goal of this project is to facilitate data-driven procurement decisions that balance cost, quality, reliability, and operational efficiency.

The sourcing decisions will be broken down into various categories including:
1. Fruit and vegetables - highly perishable
2. Non-perishables - dry goods, canned goods, packaged
3. Non-alcoholic beverages
4. Alcoholic beverages
5. Protein and dairy - meat, poultry, milk, dairy products
6. Transportation mode - delivery vs pickup

The products categories will have 4 competing suppliers

***Decision Objective***
The primary objective is to select the optimal supplier per category and the optimal mode of transportation that maximizes overall procurement utility under a balanced decision strategy. This means not one criterion dominates the decision over others. The criteria include cost efficiency, product quality, supplier reliability, and operational feasibility.

***Decision Framework***
The project will be modeled as a Multi-Creteria Decsion Analysis (MCDA) problem using a Weighted Scoring/ Utility-Based Model. The proposed weights at a balanced baseline are as follows:
      Overall Cost - 0.3
      Quality - 0.25
      Reliability and lead time - 0.25
      Operational fit - 0.1
      Sustainability - 0.1
      **Total - 1.00**





**Data Design**
Each category will have its own supplier set for instance fruits and vegetables will consist of FV_Supplier_1 ... FV_Supplier_4.

| Variable               | Description            | Rationale           |
| ---------------------- | ---------------------- | ------------------- |
| `supplier_id`          | Unique supplier name   | Identification      |
| `category`             | Procurement category   | Grouping            |
| `unit_price_gbp`       | Avg price per unit (£) | Cost driver         |
| `quality_score`        | 1–10                   | Product consistency |
| `reliability_score`    | 0–1                    | On-time fulfillment |
| `lead_time_days`       | Days to delivery       | Operations          |
| `distance_km`          | Supplier distance      | Transport & risk    |
| `min_order_qty`        | Minimum order size     | Flexibility         |
| `sustainability_score` | 1–10                   | ESG proxy           |

There are additional variable dependent on Category-specific behavior. These include:
      Fruits and vegetables - spoilage_risk (0-1)
      Protein and dairy - cold_chain_risk (0-1)

The transportation variables will be modeled per supplier:
| Variable                   | Description              |
| -------------------------- | ------------------------ |
| `transport_mode`           | Delivery / Pickup        |
| `transport_cost_gbp`       | Cost per order           |
| `transport_lead_time_days` | Added delay              |
| `transport_risk`           | Delay/spoilage risk      |
| `pickup_feasible`          | Boolean (distance-based) |




In [ ]:
import pandas as pd

file_path = '/catering_decision_project_data.xlsx'

suppliers_df = pd.read_excel(
    file_path,
    sheet_name="Suppliers"
)

transport_df = pd.read_excel(
    file_path,
    sheet_name="Transport"
)

# Clean column names
suppliers_df.columns = suppliers_df.columns.str.strip().str.lower()
transport_df.columns = transport_df.columns.str.strip().str.lower()

# Merge
df = suppliers_df.merge(
    transport_df,
    on="supplier_id",
    how="inner"
)


df.columns = df.columns.str.strip().str.lower()

print(df.head())

     supplier_id             category  unit_price_gbp  quality_score  \
0  FR_Supplier_1  Fruits & Vegetables            7.68            8.6   
1  FR_Supplier_2  Fruits & Vegetables            4.49            7.2   
2  FR_Supplier_3  Fruits & Vegetables            7.60            6.0   
3  FR_Supplier_3  Fruits & Vegetables            7.60            6.0   
4  FR_Supplier_4  Fruits & Vegetables            4.31            7.0   

   reliability_score  lead_time_days  distance_km  min_order_qty  \
0               0.93               4          112             30   
1               0.87               3           96             30   
2               0.98               4           11             10   
3               0.98               4           11             10   
4               0.93               5           31             20   

   sustainability_score  spoilage_risk transport_mode  transport_cost_gbp  \
0                   6.8           0.90       Delivery               47.40   
1   

In [ ]:
# Creating decision aggregates

df["total_cost"] = df["unit_price_gbp"] + df["transport_cost_gbp"]

df["total_lead_time"] = (
    df["lead_time_days"] + df["transport_lead_time_days"]
)

df["total_risk"] = (
    df["transport_risk"] + df["spoilage_risk"].fillna(0)
)



In [ ]:
# Defining benefits vs cost criteria

benefit_cols = [
    "quality_score",
    "reliability_score",
    "sustainability_score"
]

cost_cols = [
    "total_cost",
    "total_lead_time",
    "total_risk",
    "distance_km",
    "min_order_qty"
]


In [ ]:
# Normalizing the data

def normalize(series, benefit=True):
    if benefit:
        return (series - series.min()) / (series.max() - series.min())
    else:
        return (series.max() - series) / (series.max() - series.min())

for col in benefit_cols:
    df[f"{col}_norm"] = normalize(df[col], benefit=True)

for col in cost_cols:
    df[f"{col}_norm"] = normalize(df[col], benefit=False)


In [ ]:
# Applying balanced weights

weights = {
    "quality_score_norm": 0.25,
    "reliability_score_norm": 0.25,
    "sustainability_score_norm": 0.10,
    "total_cost_norm": 0.30,
    "total_lead_time_norm": 0.05,
    "total_risk_norm": 0.03,
    "distance_km_norm": 0.01,
    "min_order_qty_norm": 0.01
}

#Computing utility score

df["utility_score"] = sum(
    df[col] * weight for col, weight in weights.items()
)


In [ ]:
# Ranking decisions per category

df["rank"] = (
    df.groupby("category")["utility_score"]
      .rank(ascending=False, method="dense")
)

df.sort_values(["category", "rank"]).head(10)


,supplier_id,category,unit_price_gbp,quality_score,reliability_score,lead_time_days,distance_km,min_order_qty,sustainability_score,spoilage_risk,...,quality_score_norm,reliability_score_norm,sustainability_score_norm,total_cost_norm,total_lead_time_norm,total_risk_norm,distance_km_norm,min_order_qty_norm,utility_score,rank
33,AL_Supplier_1,Alcoholic Beverages,6.09,8.6,0.95,5,20,10,8.7,NaN,...,0.764706,0.769231,1.000000,0.888475,0.8,0.923077,0.910891,1.00,0.836828,1.0
32,AL_Supplier_1,Alcoholic Beverages,6.09,8.6,0.95,5,20,10,8.7,NaN,...,0.764706,0.769231,1.000000,0.573108,0.7,0.913462,0.910891,1.00,0.736929,2.0
38,AL_Supplier_4,Alcoholic Beverages,6.45,8.0,0.95,5,56,30,5.2,NaN,...,0.588235,0.769231,0.054054,0.723337,0.8,0.932692,0.554455,0.50,0.640298,3.0
36,AL_Supplier_3,Alcoholic Beverages,6.24,8.2,0.97,7,97,10,6.5,NaN,...,0.647059,0.923077,0.405405,0.127294,0.5,0.961538,0.148515,1.00,0.536594,4.0
37,AL_Supplier_4,Alcoholic Beverages,6.45,8.0,0.95,5,56,30,5.2,NaN,...,0.588235,0.769231,0.054054,0.356365,0.6,0.923077,0.554455,0.50,0.519918,5.0
35,AL_Supplier_2,Alcoholic Beverages,7.03,7.6,0.86,9,50,30,6.2,NaN,...,0.470588,0.076923,0.324324,0.732511,0.4,0.951923,0.613861,0.50,0.448760,6.0
34,AL_Supplier_2,Alcoholic Beverages,7.03,7.6,0.86,9,50,30,6.2,NaN,...,0.470588,0.076923,0.324324,0.374140,0.2,1.000000,0.613861,0.50,0.332691,7.0
3,FR_Supplier_3,Fruits & Vegetables,7.60,6.0,0.98,4,11,10,6.2,0.47,...,0.000000,1.000000,0.324324,0.883888,0.9,0.490385,1.000000,1.00,0.627310,1.0
5,FR_Supplier_4,Fruits & Vegetables,4.31,7.0,0.93,5,31,20,5.2,0.71,...,0.294118,0.615385,0.054054,0.892202,0.8,0.269231,0.801980,0.75,0.564038,2.0
2,FR_Supplier_3,Fruits & Vegetables,7.60,6.0,0.98,4,11,10,6.2,0.47,...,0.000000,1.000000,0.324324,0.581422,0.8,0.519231,1.000000,1.00,0.532436,3.0


In [ ]:
# Final optimal decisiion

df["rank"] = (
    df.groupby("category")["utility_score"]
      .rank(ascending=False, method="dense")
)

df.sort_values(["category", "rank"]).head(10)


,supplier_id,category,unit_price_gbp,quality_score,reliability_score,lead_time_days,distance_km,min_order_qty,sustainability_score,spoilage_risk,...,quality_score_norm,reliability_score_norm,sustainability_score_norm,total_cost_norm,total_lead_time_norm,total_risk_norm,distance_km_norm,min_order_qty_norm,utility_score,rank
33,AL_Supplier_1,Alcoholic Beverages,6.09,8.6,0.95,5,20,10,8.7,NaN,...,0.764706,0.769231,1.000000,0.888475,0.8,0.923077,0.910891,1.00,0.836828,1.0
32,AL_Supplier_1,Alcoholic Beverages,6.09,8.6,0.95,5,20,10,8.7,NaN,...,0.764706,0.769231,1.000000,0.573108,0.7,0.913462,0.910891,1.00,0.736929,2.0
38,AL_Supplier_4,Alcoholic Beverages,6.45,8.0,0.95,5,56,30,5.2,NaN,...,0.588235,0.769231,0.054054,0.723337,0.8,0.932692,0.554455,0.50,0.640298,3.0
36,AL_Supplier_3,Alcoholic Beverages,6.24,8.2,0.97,7,97,10,6.5,NaN,...,0.647059,0.923077,0.405405,0.127294,0.5,0.961538,0.148515,1.00,0.536594,4.0
37,AL_Supplier_4,Alcoholic Beverages,6.45,8.0,0.95,5,56,30,5.2,NaN,...,0.588235,0.769231,0.054054,0.356365,0.6,0.923077,0.554455,0.50,0.519918,5.0
35,AL_Supplier_2,Alcoholic Beverages,7.03,7.6,0.86,9,50,30,6.2,NaN,...,0.470588,0.076923,0.324324,0.732511,0.4,0.951923,0.613861,0.50,0.448760,6.0
34,AL_Supplier_2,Alcoholic Beverages,7.03,7.6,0.86,9,50,30,6.2,NaN,...,0.470588,0.076923,0.324324,0.374140,0.2,1.000000,0.613861,0.50,0.332691,7.0
3,FR_Supplier_3,Fruits & Vegetables,7.60,6.0,0.98,4,11,10,6.2,0.47,...,0.000000,1.000000,0.324324,0.883888,0.9,0.490385,1.000000,1.00,0.627310,1.0
5,FR_Supplier_4,Fruits & Vegetables,4.31,7.0,0.93,5,31,20,5.2,0.71,...,0.294118,0.615385,0.054054,0.892202,0.8,0.269231,0.801980,0.75,0.564038,2.0
2,FR_Supplier_3,Fruits & Vegetables,7.60,6.0,0.98,4,11,10,6.2,0.47,...,0.000000,1.000000,0.324324,0.581422,0.8,0.519231,1.000000,1.00,0.532436,3.0


In [ ]:
best_outcomes = df[df["rank"] == 1]
best_outcomes[["category", "supplier_id", "utility_score", "transport_mode"]]


,category,supplier_id,utility_score,transport_mode
3,Fruits & Vegetables,FR_Supplier_3,0.627310,Pickup
11,Non-Perishables,NO_Supplier_2,0.877159,Pickup
22,Non-Alcoholic Beverages,NO_Supplier_1,0.798799,Pickup
33,Alcoholic Beverages,AL_Supplier_1,0.836828,Pickup
42,Protein & Dairy,PR_Supplier_3,0.794176,Pickup


**Conclusion**

Optimal suppliers were not chosen based on the lowest-cost provider. The model focused on suppliers that performed cosnistently across multiple dimensions.

***Transport mode***

The model consistently selected Pickup over delivery mode of transportation. Pickup was chosen because:
1. It reduced total cost sufficiently
2. It did not significanlty worsen lead times
3. It likely reduced transport-related risk
4. Distance penalties were managable

The results signal that that internalizing transport (pickup strategy) is more economical and operationally efficient.

***Category-specific observations***

*Fruits and Vegetables*

FR_Supplier_3 was selected as the best option despite it not being the cheapest one. Supplier 3 is hilghy reliable and has a short lead time likely offseting the price premium. The most significant areas of evaluation in this category were supply sustainability and reduced spoilage risk since the products are perishable.

*Non-perishable*

NO_Supplier_2 had the highest overall utility score indicating a strong performance across cost, logistics, and reliablity dimensions.

*Non-Alcoholic Beverages*

NO_Supplier_1 had the best balanced trade-off between cost and sustainability without compromising on reliability.

*Alcoholic Bevarages*

AL_Supplier_1 demonstrated storng sustainability and reliablity metrics, which elevated its overall composite utility.

*Protein and Dairy*

PR_Supplier_3 emerged as optimal due to strong lead time and reliability performance which are crucial since the products are temperature-sensitive.

**Model limitations**

Although the model is well established on merit and analytically sound, there are some limitations to the project. They include:
1. Synthetic supplier data
2. No capacity constraints modeled
3. No volume discounts included
4. No multi-sourcing strategy exploerd
5. No fuel price volatility modeled

In future analyses, model interations should include stochastic demand since catering businesses have seasonal spikes, event-driven volatility, sudden cancellations, and weather-driven changes